In [1]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
import plotly.io as pio
import pickle

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

nltk.download('stopwords')
stop_words_id = stopwords.words('indonesian')

c:\skripsi\model\BERTopic_skripsi\BERTopic_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
df = pd.read_csv('data_BERTOPIC/data_BERTopic.csv')
docs = df['final_bertopic'].tolist()
print(f"Jumlah dokumen: {len(docs)}")

Jumlah dokumen: 2116


In [3]:
embedding_model = SentenceTransformer('distiluse-base-multilingual-cased')

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 966.04it/s, Materializing param=transformer.layer.5.sa_layer_norm.weight]   


In [4]:
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

hdbscan_model = HDBSCAN(min_cluster_size=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

vectorizer_model = CountVectorizer(stop_words=stop_words_id, ngram_range=(1,2))

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=True,
    verbose=True
)

In [5]:
topics, probs = topic_model.fit_transform(docs)

2026-03-02 22:18:28,657 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 67/67 [00:30<00:00,  2.17it/s]
2026-03-02 22:18:59,548 - BERTopic - Embedding - Completed ✓
2026-03-02 22:18:59,549 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-02 22:19:13,246 - BERTopic - Dimensionality - Completed ✓
2026-03-02 22:19:13,247 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-02 22:19:13,399 - BERTopic - Cluster - Completed ✓
2026-03-02 22:19:13,402 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-02 22:19:13,621 - BERTopic - Representation - Completed ✓


In [6]:
topic_info = topic_model.get_topic_info()
print(topic_info.head(10))

   Topic  Count                                          Name  \
0     -1    840                    -1_etanol_bbm_bahan_bahlil   
1      0    194            0_etanol_bahlil etanol_bahlil_pake   
2      1    106                      1_etanol_grok_bbm_campur   
3      2     73                    2_air_tangki_etanol_bensin   
4      3     63  3_etanol_mdy_asmara1701_bahlil_bahlil etanol   
5      4     63                     4_etanol 10_10_etanol_bbm   
6      5     37             5_indonesia_etanol_bbm_penggunaan   
7      6     33                        6_rakyat_bbm_etanol_gk   
8      7     32                 7_pertamina_shell_swasta_spbu   
9      8     32                  8_e10_kendaraan_aman_e10 e20   

                                      Representation  \
0  [etanol, bbm, bahan, bahlil, bakar, 10, bensin...   
1  [etanol, bahlil etanol, bahlil, pake, si, pake...   
2  [etanol, grok, bbm, campur, campur etanol, bbm...   
3  [air, tangki, etanol, bensin, bbm, campur, hig...   
4  [

In [7]:
topic_model.get_topic(1)

[('etanol', np.float64(0.05204182128483627)),
 ('grok', np.float64(0.042676171329845934)),
 ('bbm', np.float64(0.028385164880449273)),
 ('campur', np.float64(0.021411515927799583)),
 ('campur etanol', np.float64(0.01947561404022821)),
 ('bbm etanol', np.float64(0.018978417152023547)),
 ('si', np.float64(0.018883071847881593)),
 ('kah', np.float64(0.017182658259756532)),
 ('bahlillahadalia', np.float64(0.016916825591053217)),
 ('bahlil', np.float64(0.014480198316075613))]

In [8]:
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

tokenized_docs = [doc.split() for doc in docs]

dictionary = Dictionary(tokenized_docs)
corpus = [dictionary.doc2bow(tokens) for tokens in tokenized_docs]

topics_words = []
for topic_id in set(topics):
    if topic_id != -1:  # -1 adalah outlier
        words = [word for word, _ in topic_model.get_topic(topic_id)[:10]]
        topics_words.append(words)

if topics_words:
    cv_model = CoherenceModel(topics=topics_words, texts=tokenized_docs, dictionary=dictionary, coherence='c_v')
    cv_score = cv_model.get_coherence()
    
    npmi_model = CoherenceModel(topics=topics_words, texts=tokenized_docs, dictionary=dictionary, coherence='c_npmi')
    npmi_score = npmi_model.get_coherence()
    
    print(f"C_v Coherence: {cv_score:.4f}")
    print(f"C_npmi Coherence: {npmi_score:.4f}")
else:
    print("Tidak ada topik yang terbentuk.")

C_v Coherence: 0.5281
C_npmi Coherence: 0.0560


In [9]:
fig_barchart = topic_model.visualize_barchart(top_n_topics=10, n_words=10)
pio.write_html(fig_barchart, 'visualisasi/barchart_topik.html')

In [10]:
fig_heatmap = topic_model.visualize_heatmap(n_clusters=5, top_n_topics=10)
pio.write_html(fig_heatmap, 'visualisasi/heatmap_topik.html')

In [11]:
fig_hierarchy = topic_model.visualize_hierarchy(top_n_topics=10)
pio.write_html(fig_hierarchy, 'visualisasi/hierarchy_topik.html')

In [12]:
fig_dist = topic_model.visualize_distribution(probs[0])  # untuk dokumen pertama
pio.write_html(fig_dist, 'visualisasi/distribusi_dokumen.html')

In [13]:
with open('model/bertopic_model.pkl', 'wb') as f:
    pickle.dump(topic_model, f)

with open('model/bertopic_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

AttributeError: Can't pickle local object 'install_output_capuring_hook.<locals>.output_capturing_hook'